# DATA ANALYSIS


Total data in the WM-881K wafermap dataset is 811457,,,but There are only 172.950 data labelled, and among 9 categories there is a big imbalance among distribution of data....['none','Loc','Edge-Loc','Center','Edge-Ring','Scratch','Random','Near-full','Donut'] == [147431, 3593, 5189, 4294, 9680, 1193, 866, 149, 555] data in different category,,,As you can see, the dataset is strongly biased towards the none class, that is the one without defects. We will definitely need to do data augmentation.not all the images have the same shape. It will therefore be necessary to resize them before being able to feed them to a neural network.There are 346 different shapes assumed by the various images.Mean shape of those image 
(57.73699421965318, 57.26589595375722) and median shape (53,52). To give all the images the same shape, we choose to bring them back to the median dimension of the dataset which, as already seen, also coincides with one of the 346 shapes assumed by the images.Given the substantial square shape of the median shape, we check if there are images with a particularly rectangular shape that we could exclude from the dataset. Specifically, the constraint is that one of the two dimensions is at least 50% larger than the other.

 

# DATA CLEANING

In this phase we clean up the dataset of defective or excessively rectangular images.Let's get rid of excessively rectangular images first.We divide the images by shape and we count, for each shape, the average proportion between pixels reserved for the chip and pixels reserved for the background.
Let's see first which are the shapes with the greater average background pixel.Let's save the indices of the defective images for each shape, i.e. with very low non-background pixel ratio (less than 0.6).We display the defective images (proportion < 0.6).Instead, we observe that for images with a proportion greater than 0.6 the shape of the wafer is well identifiable.There are 22 defective images. We save the indexes.We eliminate the defective images.Now we have the dataset cleansed of flawed and unbalanced images. Let's save it.


# DATA PREPARATION

Once we have cleaned the dataset of defective images, we can work with the remaining 170246 images. First of all, we eliminate the useless columns, i.e. dieSize, lotName, waferIndex, trainTestLabel, shape.

The two remaining columns represent the image pixels and the type of anomaly, one of 9 classes: Center, Donut, Edge-Loc, Edge-Ring, Loc, Near-full, Random, Scratch, none. The pixels instead can assume 3 possible values:

    0: Background;
    1: Normal chip;
    2: Abnormal chip.

The classes are very unbalanced, so we will need to do data augmentation.

Finally, in this phase we divide the dataset between training-set (80%) and test-set (20%),,,,,

# DATA AUGMENTATION

Before proceeding with augmentation, it is necessary to resize the data to the shape (224, 224) for compatibility with our CNN pretrained model. So, we apply padding to the dimensions, so that they become (224, 224) or multiples. Once the images are resized,we apply flipping and rotation to each anomalous image.We define a padding function that adds background pixels to the images, in order to make the dimensions of the new shape multiple of 224 and 224 respectively. So, the visual appearance of the images, when resizing to the shape (224, 224) has not changed.We do padding on the training data.
We do padding on the test data.Now, let's resize the padded training data. We use nearest as type of interpolation, so that the modified images still have pixels that assume value:

    0: background;
    1: chip ok;
    2: anomalous chip.

#### We apply horizontal and vertical flipping and 90° and 270° rotation to the images belonging to the  classes.
  from PIL import Image
import numpy as np
import copy

for i in range(len(training_aug) - 1):
    print(classes[i])
    for j in range(len(training_aug[i]['waferMap'])):
        # Get the wafer map
        wafer_map = np.array(training_aug[i]['waferMap'][j], dtype=np.uint8)
        
        # Check and reshape if needed
        if len(wafer_map.shape) == 3 and wafer_map.shape[-1] == 1:  # Shape (height, width, 1)
            wafer_map = wafer_map.squeeze(-1)  # Convert to (height, width)
        
        # Ensure the array is valid for image conversion
        if len(wafer_map.shape) not in [2, 3]:
            raise ValueError(f"Invalid image shape: {wafer_map.shape} at index {i}, {j}")
        
        # Create the image
        img = copy.deepcopy(Image.fromarray(wafer_map))
        
        # Augmentations
        horiz_img = img.transpose(method=Image.FLIP_LEFT_RIGHT)
        vert_img = img.transpose(method=Image.FLIP_TOP_BOTTOM)
        rotated_img_1 = img.transpose(method=Image.ROTATE_90)
        rotated_img_2 = img.transpose(method=Image.ROTATE_270)
        
        # Add augmented data back to DataFrame
        training_aug[i].loc[training_aug[i].shape[0]] = [horiz_img, classes[i]]
        training_aug[i].loc[training_aug[i].shape[0]] = [vert_img, classes[i]]
        training_aug[i].loc[training_aug[i].shape[0]] = [rotated_img_1, classes[i]]
        training_aug[i].loc[training_aug[i].shape[0]] = [rotated_img_2, classes[i]]



  
#### NOW WE MAKE A FUNCTION TO NORMALIZE OUR IMAGES AND THEN CONVERT THESE IMAGES FROM GRAY CMAP TO RGB MODE BECAUSE OUR CNN pretrained model need image shape (224,224,3) for perfect fitting,,,an then finally save train data's image in a train dataset folder in different category folder

normalize_and_convert(image):
    # Check if image has 3 channels or 1 channel
    if len(image.shape) == 3 and image.shape[2] == 1:
        # Remove singleton dimension
        image_2d = image.squeeze(axis=2)
    elif len(image.shape) == 2:
        # If the image is already 2D, use it directly
        image_2d = image
    else:
        raise ValueError("Invalid image shape")

    # Normalize to [0, 255]
    min_val, max_val = image_2d.min(), image_2d.max()
    if max_val - min_val == 0:
        image_normalized = np.zeros_like(image_2d, dtype=np.uint8)
    else:
        image_normalized = ((image_2d - min_val) / (max_val - min_val) * 255).astype(np.uint8)
    
    # Convert grayscale to RGB by stacking channels
    image_rgb = cv2.cvtColor(image_normalized, cv2.COLOR_GRAY2RGB)
    
    return image_rgb

for i in range(len(classes)):
    class_name = classes[i]
    class_dir = os.path.join(path, class_name)
    
    for j in range(len(training_aug[i])):
        # Define the filename
        filename = f"{j}.png"
        file_path = os.path.join(class_dir, filename)
        
        # Print progress every 1000 images
        if j % 1000 == 0:
            print(f"Saving: {file_path}")
        
        # Extract the image
        image = training_aug[i]['waferMap'].iloc[j]
        
        # Convert and normalize the image
        try:
            image_rgb = normalize_and_convert(image)
        except ValueError as e:
            print(f"Error processing image at class '{class_name}', index {j}: {e}")
            continue  # Skip this image and continue
        
        # Save the image using OpenCV
        success = cv2.imwrite(file_path, image_rgb)
        if not success:
            print(f"Failed to save image: {file_path}")

we do same for test data also .....

#### now we load our train dataset and test dataset from their respective folder with image_dataset_from_directory kera function :

batch_size = 32
img_height = 224
img_width = 224
seed = 123
train_dir = 'WM811K_train_split'
val_dir = 'WM811K_val_split'
  #3 Load Training Dataset
train_dataset = keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=(img_height, img_width),
    shuffle=True,
    seed=seed
 )

  #3 Load Validation Dataset
val_dataset = keras.preprocessing.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=(img_height, img_width),
    shuffle=False,
    seed=seed
)


##### and we see data are imbalanced in different classes or train and test data,,like train data classes have these images 
['Center', 'Donut','Edge-Loc', 'Edge-Ring','Loc','Near-full','Random', 'Scratch','none'] = [17115, 15000, 20170, 38640, 15000, 15000, 15000, 15000, 116022]  where none category has huge image data of 116022 and center,donut,Loc,Random etc has very low data,,,
so we used this function to remove our own calculated image data from different category inorder to make balanced data in every category 
code:
-----
#3 Define paths
train_aug_path = 'WM811K_train_split'
category_to_remove = 'none'  # Replace with the actual category name you want to remove images from

#3 Path to the folder of the category you want to remove images from
category_folder_path = os.path.join(train_aug_path, category_to_remove)

#3 List all files in the category folder
image_files = os.listdir(category_folder_path)

#3 Specify the number of images to remove (e.g., remove 5 random images)
num_images_to_remove = 10000 # Adjust this to your desired number

#3 Ensure we don't try to remove more images than exist in the category
num_images_to_remove = min(num_images_to_remove, len(image_files))

#3 Randomly select a subset of images to remove
images_to_remove = random.sample(image_files, num_images_to_remove)

#3 Remove the selected images
for image in images_to_remove:
    image_path = os.path.join(category_folder_path, image)
    
    # Remove the image file
    os.remove(image_path)
    print(f"Removed {image_path}")


and we do same for test data and finally we load them with the help if image_dataset_from_directory
and finally get Found 106492 files belonging to 9 classes for train data which is now balanced
and Found 26623 files belonging to 9 classes for test data which is also balanced

and Finally train our model......

# RESNET50 MODEL CODE


In [1]:
from tensorflow.keras.applications import ResNet50



# Load ResNet50 base model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the base model initially

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)  # Dropout to reduce overfitting
x = Dense(1024, activation='relu', kernel_regularizer=l2(0.01))(x)
x = Dropout(0.3)(x)
predictions = Dense(len(train_aug_dataset.class_names), activation='softmax')(x)

# Create the final model
resnet0 = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
resnet0.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

# Compute class weights to handle class imbalance
y_train_classes = np.concatenate([y.numpy() for _, y in train_aug_dataset])
y_train_labels = np.argmax(y_train_classes, axis=1)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weights = dict(enumerate(class_weights))

# Callbacks for training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ModelCheckpoint('resnetbest20l.keras', monitor='val_loss', save_best_only=True, mode='min'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

# Train only the top layers (before fine-tuning)
hres020l = resnet0.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks,
    class_weight=class_weights
)


# Unfreeze some layers in the base model for fine-tuning
base_model.trainable = True
# Freeze all layers except the last 5
for layer in base_model.layers[:-15]:
    layer.trainable = False

# Recompile the model with a lower learning rate for fine-tuning
resnet0.compile(optimizer=Adam(learning_rate=1e-4),
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

# Fine-tune the model
historyfineres20l = resnet0.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    # Remove batch_size if train_aug_dataset is already batched
    callbacks=callbacks,
    class_weight=class_weights
)






# from tensorflow.keras.models import load_model
# model_path = "resnetbest20l.keras"
# # Load the model
# resnet0 = load_model(model_path)

In [2]:
resnet0.summary()

Model: "model_14"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_14 (InputLayer)          [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_14[0][0]']               
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

# XAI GRADCAM CODE

In [ ]:
def save_layer_wise_gradcam(val_dataset, model, conv_layer_names, class_names, save_dir, num_samples=1):
    """
    Generates and saves layer-wise Grad-CAM visualizations for specified convolutional layers.

    Parameters:
    - val_dataset: Validation dataset.
    - model: Trained CNN model.
    - conv_layer_names: List of convolutional layer names to visualize.
    - class_names: List of class names corresponding to label indices.
    - save_dir: Directory to save the visualization images.
    - num_samples: Number of samples to visualize.
    """
    layer_dir = os.path.join(save_dir, "layer_wise_gradcam")
    os.makedirs(layer_dir, exist_ok=True)

    for layer_name in conv_layer_names:
        layer_specific_dir = os.path.join(layer_dir, layer_name)
        os.makedirs(layer_specific_dir, exist_ok=True)

    sample_count = 0

    for batch in val_dataset:
        images, labels = batch
        preds = model.predict(images)
        pred_classes = np.argmax(preds, axis=1)

        for i in range(images.shape[0]):
            img = images[i].numpy()
            img_uint8 = (img * 255).astype('uint8') if img.max() <= 1.0 else img.astype('uint8')

            # Expand dimensions and preprocess the image
            img_array = np.expand_dims(img, axis=0)
            preprocessed_img = tf.keras.applications.resnet50.preprocess_input(img_array.copy())

            # Predict class
            pred_class = pred_classes[i]

            for layer_name in conv_layer_names:
                # Generate heatmap
                heatmap = make_gradcam_heatmap(preprocessed_img, model, layer_name, pred_index=pred_class)

                # Overlay heatmap
                heatmap_colored, superimposed_img = overlay_heatmap_on_image(img_uint8, heatmap)

                # Plot and save the images
                plt.figure(figsize=(15, 5))

                plt.subplot(1, 3, 1)
                plt.imshow(img_uint8)
                plt.title(f"Original Image\nPred: {class_names[pred_class]}")
                plt.axis('off')

                plt.subplot(1, 3, 2)
                plt.imshow(heatmap, cmap='jet')
                plt.title(f"Grad-CAM Heatmap\nLayer: {layer_name}")
                plt.axis('off')

                plt.subplot(1, 3, 3)
                plt.imshow(superimposed_img)
                plt.title(f"Overlay\nLayer: {layer_name}")
                plt.axis('off')

                plt.tight_layout()
                sample_filename = f"{layer_name}_sample_{sample_count + 1}.png"
                plt.savefig(os.path.join(layer_specific_dir, sample_filename))
                plt.close()

            sample_count += 1
            if sample_count >= num_samples:
                return  # Stop after saving the desired number of samples


# Define the convolutional layers you want to visualize
conv_layer_names = [
    'conv5_block3_out',  # Final block of conv5
    'conv4_block6_out',  # Final block of conv4
    'conv3_block4_out',  # Final block of conv3 (added)
    'conv2_block3_out'   # Final block of conv2 (added)
]  # Replace these with actual layer names if different in your ResNet50 model

# Save layer-wise Grad-CAM visualizations
save_layer_wise_gradcam(
    val_dataset=val_dataset,
    model=resnet0,
    conv_layer_names=conv_layer_names,
    class_names=class_names,
    save_dir=save_dir,
    num_samples=2  # Adjust as needed
)

print("Layer-wise Grad-CAM visualizations have been saved.")


# (OPTIONAL PART)
## CUSTOM MODEL CODE

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import callbacks

# Define the number of classes
num_classes = 9

# Input layer
inputs = keras.Input(shape=(224, 224, 3))

# He Normal initializer
he_init = tf.keras.initializers.HeNormal(seed=42)

# Block 1: Conv -> BN -> ReLU -> MaxPool
x = layers.Conv2D(32, kernel_size=3, padding='same', kernel_initializer=he_init)(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D(pool_size=2, padding='same')(x)

# Block 2: Conv -> BN -> ReLU -> MaxPool
x = layers.Conv2D(64, kernel_size=3, padding='same', kernel_initializer=he_init)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D(pool_size=2, padding='same')(x)

# Block 3: Conv -> BN -> ReLU -> Global Pool
x = layers.Conv2D(128, kernel_size=3, padding='same', kernel_initializer=he_init)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.GlobalAveragePooling2D()(x)

# Fully Connected Layers
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

# Output layer
outputs = layers.Dense(num_classes, activation='softmax')(x)

# Define the model
modelcustom = keras.Model(inputs, outputs)

# Compile the model with an optimized optimizer
modelcustom.compile(
    loss=keras.losses.CategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

# Define callbacks
callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ModelCheckpoint('custom_model.keras', monitor='val_loss', save_best_only=True, mode='min')
]



# Train the model
hcustom = modelcustom.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks_list
)

## INCEPTION V3 CODE

In [ ]:
from tensorflow.keras.applications import InceptionV3

# Load InceptionV3 base model
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))  # InceptionV3 uses 299x299 input size
base_model.trainable = False  # Freeze the base model initially

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)  # Dropout to reduce overfitting
x = Dense(1024, activation='relu', kernel_regularizer=l2(0.01))(x)
x = Dropout(0.3)(x)
predictions = Dense(len(train_dataset.class_names), activation='softmax')(x)

# Create the final model
inception_model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
inception_model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

# Compute class weights to handle class imbalance
y_train_classes = np.concatenate([y.numpy() for _, y in train_dataset])
y_train_labels = np.argmax(y_train_classes, axis=1)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weights = dict(enumerate(class_weights))

# Callbacks for training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ModelCheckpoint('inceptionbest.keras', monitor='val_loss', save_best_only=True, mode='min'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

# Train only the top layers (before fine-tuning)
history_initial = inception_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks,
    class_weight=class_weights
)



# Unfreeze some layers in the base model for fine-tuning
base_model.trainable = True
# Freeze all layers except the last 5
for layer in base_model.layers[:-15]:
    layer.trainable = False

# Recompile the model with a lower learning rate for fine-tuning
inception_model.compile(optimizer=Adam(learning_rate=1e-4),
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

# Fine-tune the model
hincepfine = inception_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    # Remove batch_size if train_aug_dataset is already batched
    callbacks=callbacks,
    class_weight=class_weights
)

# Evaluate on the test dataset
test_loss, test_accuracy = inception_model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


## SVM CODE

In [ ]:
# ---multicalss classification ---#
# One-Vs-One
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsOneClassifier
clf2 = OneVsOneClassifier(LinearSVC(random_state = RANDOM_STATE)).fit(X_train, y_train)
y_train_pred = clf2.predict(X_train)
y_test_pred = clf2.predict(X_test)
train_acc2 = np.sum(y_train == y_train_pred, axis=0, dtype='float') / X_train.shape[0]
test_acc2 = np.sum(y_test == y_test_pred, axis=0, dtype='float') / X_test.shape[0]
print('One-Vs-One Training acc: {}'.format(train_acc2*100)) #One-Vs-One Training acc: 80.36
print('One-Vs-One Testing acc: {}'.format(test_acc2*100)) #One-Vs-One Testing acc: 79.04
print("y_train_pred[:100]: ", y_train_pred[:100])
print ("y_train[:100]: ", y_train[:100])